<img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:100px" alt="MCUNet Logo">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Déploiement TinyML avec MCUNet </h1></center>
<center><h2> Notebook 04 : Validation du déploiement avec TinyEngine (Solution) </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

>
> **Objectif :** Valider qu'un modèle est réellement compatible avec un MCU cible en utilisant TinyEngine pour l'analyse statique de la mémoire.
>
> **Durée estimée :** 40 minutes
>
> **Prérequis :** Comprendre les métriques SRAM/Flash et avoir un fichier `.tflite`.
>


### Concepts clés
>
> 1. **TinyEngine** : moteur d'inférence de MCUNet. Contrairement à TFLite Micro qui est un *interpréteur* fonctionnant à l'exécution, TinyEngine est un *générateur de code*. Il analyse le graphe, planifie la mémoire à l'avance (Ahead-Of-Time), et génère du code C pur ultra-optimisé.
> 2. **Memory Scheduling** : méthode consistant à réutiliser les mêmes blocs de SRAM pour différents tenseurs à différents moments pour minimiser le pic de mémoire.


--- 
### 1. Analyse statique avec TinyEngine

Dans un projet complet, nous compilerions l'outil en C++ de TinyEngine. Pour ce cours, nous avons créé un wrapper Python (`scripts/tinyengine_analysis.py`) qui simule cette analyse statique (parsing du graphe et estimation de l'ordonnancement mémoire).

In [ ]:
import os
import sys
sys.path.append('/workspace/scripts/')
from tinyengine_analysis import analyze_tflite, print_report

# Analysons le modèle mcunet-vww1
tflite_vww1 = os.path.expanduser("~/.mcunet/mcunet-vww1.tflite")
cible_mcu = "STM32F746"

print(f"Lancement de l'analyse pour {cible_mcu}...")
result_vww1 = analyze_tflite(tflite_vww1)
print_report(result_vww1, cible_mcu)

Le modèle `mcunet-vww1` passe sans problème ! Le pic SRAM calculé est cohérent avec le Model Zoo (autour de 160 KB).

--- 
### 2. Petit exercice pratique : tester les limites

Que se passe-t-il si nous essayons de déployer `mcunet-vww2` sur ce même MCU ?
Utiliser la même méthode pour l'analyser.

In [ ]:
# Solution
tflite_vww2 = os.path.expanduser("~/.mcunet/mcunet-vww2.tflite")

print(f"Lancement de l'analyse pour mcunet-vww2 sur {cible_mcu}...")
result_vww2 = analyze_tflite(tflite_vww2)
print_report(result_vww2, cible_mcu)


Débordement mémoire. Le modèle ne pourra pas être flashé sur ce microcontrôleur.

--- 
### Points d'attention

<div class="alert alert-warning">
<i class='fa fa-exclamation-triangle '></i> &emsp; 
  <strong>Taille du fichier ≠ Taille en SRAM.</strong> Une erreur classique est de regarder la taille du fichier <code>.tflite</code> sur le disque pour savoir si le modèle va "tenir". La taille du fichier correspond (à peu près) à la mémoire <strong>Flash</strong> requise. Le pic SRAM, lui, est invisible sans faire une analyse du graphe et des tailles d'activations, comme le fait TinyEngine.
</div>

--- 
### Questions de réflexion

1. Notre analyseur simule un Memory Scheduler. Pourquoi le vrai TinyEngine parvient-il souvent à obtenir un pic SRAM légèrement inférieur à notre estimation théorique basée sur les tenseurs TFLite standard ?

--> TinyEngine fait de la fusion d'opérateurs (In-Place Updates) de manière très agressive. Si la sortie d'une opération peut directement écraser son entrée en mémoire sans altérer le calcul, le pic SRAM de cette couche est divisé par deux. L'analyseur TFLite standard ne voit pas cette optimisation.

2. Quel est l'avantage principal de générer du code C statique (TinyEngine) par rapport à embarquer un interpréteur complet (TFLite Micro) sur le MCU ?

--> Le code statique n'a pas besoin d'allouer dynamiquement la mémoire à l'exécution, il n'a pas l'overhead (Flash et SRAM) du moteur TFLite lui-même, et les boucles for sont déroulées et optimisées spécifiquement pour l'architecture cible à la compilation. C'est plus léger et beaucoup plus rapide.